# 07 — Data Collection and Cleaning (Extra Indicators)

## Purpose

This notebook is a **feasibility investigation**, not a pipeline step. It
answers a specific question:

> Are there additional World Bank indicators that could improve the
> model's predictive performance?

## Candidate indicators considered

Five additional indicators were pulled from the World Bank WDI database
for the EU-27 panel (1995–2025):

| Code | Name | Category |
|---|---|---|
| `FS.AST.PRVT.GD.ZS` | Domestic credit to private sector (% GDP) | Financial depth |
| `BN.CAB.XOKA.GD.ZS` | Current account balance (% GDP) | External balance |
| `FM.LBL.BMNY.GD.ZS` | Broad money (% GDP) | Financial depth |
| `TT.PRI.MRCH.XD.WD` | Terms of trade index (2015 = 100) | Trade shock |
| `NY.GDP.PCAP.KD.ZG` | GDP per capita growth (annual %) | Redundant with existing features |

## Method

For each candidate, we measure **coverage** across the 27 × 31 panel
(837 country-years) before deciding whether to test it in the model.
Indicators with structural gaps are rejected. Only the single viable
candidate is carried forward into `08_feature_expansion_experiment.ipynb`.

## Output

The notebook does **not** save a modelling file. Its real output is the
loader module `src/preprocessing_extra.py`, which handles the reshape
and merge for the one viable indicator (current account balance).

In [10]:
import pandas as pd
from pathlib import Path

current = Path.cwd()
project_root = current.parent if current.name == "notebooks" else current
path = project_root / "data" / "raw" / "wdi_eu27_extra_indicators_1995_2025.csv"

## 1. Inspect the raw extra-indicators file

The World Bank CSV export uses the same wide format as the main
dataset: one row per (country, indicator), with years as columns.
The file also contains metadata footer rows that must be dropped
before processing.

### Metadata footer rows

World Bank exports append several non-data rows at the bottom of the
file (blank rows and a "Data from database" / "Last Updated" line).
We identify them by NaN in `Series Code` and drop them.

In [11]:
# Peek at the first few lines as text to see if there's a metadata header
with open(path) as f:
    for i, line in enumerate(f):
        print(f"Line {i}: {line[:120]}")
        if i >= 5:
            break

print()
df = pd.read_csv(path)
print("Shape:", df.shape)
print("Columns (first 8):", df.columns.tolist()[:8])
print("First rows:")
print(df.head(3).to_string())

Line 0: Country Name,Country Code,Series Name,Series Code,1995 [YR1995],1996 [YR1996],1997 [YR1997],1998 [YR1998],1999 [YR1999],
Line 1: Austria,AUT,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,2.66798388882503,2.21576713087359,2.1458350552646,3.49288002962309,3
Line 2: Austria,AUT,GDP per capita growth (annual %),NY.GDP.PCAP.KD.ZG,2.51091305108029,2.07784870159793,2.03015241563726,3.3793
Line 3: Austria,AUT,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,2.24336631500898,1.86097115760437,1.30597857227691,0.
Line 4: Austria,AUT,Gross fixed capital formation (% of GDP),NE.GDI.FTOT.ZS,25.6814352868846,25.9792394170832,25.5410735111136,2
Line 5: Austria,AUT,"Unemployment, total (% of total labor force) (modeled ILO estimate)",SL.UEM.TOTL.ZS,4.345,5.282,5.15,5.483,

Shape: (383, 35)
Columns (first 8): ['Country Name', 'Country Code', 'Series Name', 'Series Code', '1995 [YR1995]', '1996 [YR1996]', '1997 [YR1997]', '1998 [YR1998]']
First rows:
  Country Name Country Code                   

In [12]:
# Find rows where Series Code is NaN
junk = df[df["Series Code"].isna()]
print("Number of junk rows:", len(junk))
print()
print(junk.to_string())

Number of junk rows: 5

                                         Country Name Country Code Series Name Series Code 1995 [YR1995] 1996 [YR1996] 1997 [YR1997] 1998 [YR1998] 1999 [YR1999] 2000 [YR2000] 2001 [YR2001] 2002 [YR2002] 2003 [YR2003] 2004 [YR2004] 2005 [YR2005] 2006 [YR2006] 2007 [YR2007] 2008 [YR2008] 2009 [YR2009] 2010 [YR2010] 2011 [YR2011] 2012 [YR2012] 2013 [YR2013] 2014 [YR2014] 2015 [YR2015] 2016 [YR2016] 2017 [YR2017] 2018 [YR2018] 2019 [YR2019] 2020 [YR2020] 2021 [YR2021] 2022 [YR2022] 2023 [YR2023] 2024 [YR2024] 2025 [YR2025]
378                                               NaN          NaN         NaN         NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           

In [13]:
df_clean = df.dropna(subset=["Series Code", "Country Code"]).copy()
print("After cleaning:", df_clean.shape)

After cleaning: (378, 35)


## 2. Which indicators are in this file?

The file contains 14 indicators. Nine of them are duplicates of the
core dataset (already loaded in notebook 01), and five are new
candidates.

In [14]:
print("Shape:", df_clean.shape)
print()
print("Unique indicators in this file:")
for name, code in df_clean[["Series Name", "Series Code"]].drop_duplicates().sort_values("Series Code").values:
    print(f"  {code:30s}  {name}")
print()
print("Countries in this file:", df_clean["Country Code"].nunique())
print("Countries:", sorted(df_clean["Country Code"].unique()))

Shape: (378, 35)

Unique indicators in this file:
  BN.CAB.XOKA.GD.ZS               Current account balance (% of GDP)
  BX.KLT.DINV.WD.GD.ZS            Foreign direct investment, net inflows (% of GDP)
  FM.LBL.BMNY.GD.ZS               Broad money (% of GDP)
  FP.CPI.TOTL.ZG                  Inflation, consumer prices (annual %)
  FS.AST.PRVT.GD.ZS               Domestic credit to private sector (% of GDP)
  NE.CON.GOVT.ZS                  General government final consumption expenditure (% of GDP)
  NE.EXP.GNFS.ZS                  Exports of goods and services (% of GDP)
  NE.GDI.FTOT.ZS                  Gross fixed capital formation (% of GDP)
  NE.IMP.GNFS.ZS                  Imports of goods and services (% of GDP)
  NY.GDP.MKTP.KD.ZG               GDP growth (annual %)
  NY.GDP.PCAP.KD.ZG               GDP per capita growth (annual %)
  SL.UEM.TOTL.ZS                  Unemployment, total (% of total labor force) (modeled ILO estimate)
  SP.POP.GROW                     Population 

## 3. Coverage analysis

Before deciding whether to test any candidate indicator, we measure
its **coverage** across the 27 × 31 panel. Missingness patterns matter:
if a series is missing for all countries in early years (a structural
gap), imputation would be inventing data; if missingness is scattered,
it can be filled safely.

We compute the percentage of missing values for each candidate.

In [15]:
new_codes = [
    "FS.AST.PRVT.GD.ZS",   # domestic credit
    "BN.CAB.XOKA.GD.ZS",   # current account balance
    "FM.LBL.BMNY.GD.ZS",   # broad money
    "TT.PRI.MRCH.XD.WD",   # terms of trade
    "NY.GDP.PCAP.KD.ZG",   # gdp per capita growth
]

# Reshape to long: one row per country-year-indicator
year_cols = [c for c in df_clean.columns if "[YR" in c]
long = df_clean.melt(
    id_vars=["Country Code", "Series Code"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Value",
)
long["Year"] = long["Year"].str.extract(r"(\d{4})").astype(int)
long["Value"] = pd.to_numeric(long["Value"], errors="coerce")

# NaN counts per indicator
print("Missing values per new indicator (out of 27 countries × 31 years = 837):")
for code in new_codes:
    sub = long[long["Series Code"] == code]
    total = len(sub)
    missing = sub["Value"].isna().sum()
    print(f"  {code:22s}  {missing:4d} / {total} missing  ({100*missing/total:.1f}%)")

Missing values per new indicator (out of 27 countries × 31 years = 837):
  FS.AST.PRVT.GD.ZS        192 / 837 missing  (22.9%)
  BN.CAB.XOKA.GD.ZS         43 / 837 missing  (5.1%)
  FM.LBL.BMNY.GD.ZS        626 / 837 missing  (74.8%)
  TT.PRI.MRCH.XD.WD        297 / 837 missing  (35.5%)
  NY.GDP.PCAP.KD.ZG          0 / 837 missing  (0.0%)


### Coverage verdict

| Indicator | Missing | Verdict |
|---|---|---|
| `FS.AST.PRVT.GD.ZS` (credit) | 22.9% | ❌ structural, pre-2001 |
| `BN.CAB.XOKA.GD.ZS` (current account) | 5.1% | ✅ **test this one** |
| `FM.LBL.BMNY.GD.ZS` (broad money) | 74.8% | ❌ hopeless |
| `TT.PRI.MRCH.XD.WD` (terms of trade) | 35.5% | ❌ all pre-2005 |
| `NY.GDP.PCAP.KD.ZG` (GDP pc growth) | 0% | ⏭️ redundant |

Only **current account balance** has both acceptable coverage and
non-redundant economic information.

In [16]:
# Which years are missing for terms of trade?
print()
print("Missing values by year for terms of trade (TT.PRI.MRCH.XD.WD):")
tt = long[long["Series Code"] == "TT.PRI.MRCH.XD.WD"]
print(tt.groupby("Year")["Value"].apply(lambda s: s.isna().sum()).to_string())


Missing values by year for terms of trade (TT.PRI.MRCH.XD.WD):
Year
1995    27
1996    27
1997    27
1998    27
1999    27
2000    27
2001    27
2002    27
2003    27
2004    27
2005     0
2006     0
2007     0
2008     0
2009     0
2010     0
2011     0
2012     0
2013     0
2014     0
2015     0
2016     0
2017     0
2018     0
2019     0
2020     0
2021     0
2022     0
2023     0
2024     0
2025    27


### Why terms of trade is rejected

Terms of trade has 35.5% missing — but more importantly, the missingness
is *entirely structural*: every country is missing for 1995–2004, and
none is missing for 2005–2024. Imputation would fabricate a decade of
data.

In [17]:
# Where is credit-to-GDP missing?
credit = long[long["Series Code"] == "FS.AST.PRVT.GD.ZS"]
print("Credit-to-GDP missing by year:")
print(credit.groupby("Year")["Value"].apply(lambda s: s.isna().sum()).to_string())
print()
print("Credit-to-GDP missing by country:")
missing_by_country = (
    credit.groupby("Country Code")["Value"]
    .apply(lambda s: s.isna().sum())
    .sort_values(ascending=False)
)
print(missing_by_country.to_string())

Credit-to-GDP missing by year:
Year
1995    21
1996    20
1997    20
1998    20
1999    20
2000    20
2001     7
2002     7
2003     7
2004     5
2005     4
2006     3
2007     3
2008     3
2009     3
2010     1
2011     1
2012     0
2013     0
2014     0
2015     0
2016     0
2017     0
2018     0
2019     0
2020     0
2021     0
2022     0
2023     0
2024     1
2025    26

Credit-to-GDP missing by country:
Country Code
HRV    18
LTU    16
LVA    16
SVK    12
MLT    11
SVN    10
EST    10
DEU     8
BEL     7
FIN     7
AUT     7
PRT     7
LUX     7
IRL     7
FRA     7
CYP     7
ESP     7
ITA     7
GRC     7
NLD     7
CZE     1
BGR     1
DNK     1
HUN     1
POL     1
ROU     1
SWE     1


### Why credit-to-GDP is rejected

Credit-to-GDP has 22.9% missing. The gap is concentrated in early years
and in specific countries (Croatia, Lithuania, Latvia — newer EU members).
Structural, not random. Rejected.

In [18]:
cab = long[long["Series Code"] == "BN.CAB.XOKA.GD.ZS"]
print("Current account balance missing by year:")
print(cab.groupby("Year")["Value"].apply(lambda s: s.isna().sum()).to_string())
print()
print("Current account balance missing by country:")
print(cab.groupby("Country Code")["Value"].apply(lambda s: s.isna().sum()).sort_values(ascending=False).to_string())

Current account balance missing by year:
Year
1995    5
1996    5
1997    5
1998    6
1999    3
2000    3
2001    3
2002    2
2003    2
2004    2
2005    0
2006    0
2007    0
2008    0
2009    0
2010    0
2011    0
2012    0
2013    0
2014    0
2015    0
2016    0
2017    0
2018    0
2019    0
2020    0
2021    0
2022    0
2023    0
2024    0
2025    7

Current account balance missing by country:
Country Code
IRL    11
AUT    10
BEL     7
LUX     4
DEU     4
GRC     2
DNK     1
HRV     1
MLT     1
PRT     1
NLD     1
FRA     0
FIN     0
EST     0
ESP     0
CYP     0
CZE     0
BGR     0
HUN     0
LVA     0
LTU     0
ITA     0
POL     0
ROU     0
SVK     0
SVN     0
SWE     0


### Current account balance — viable

Current account has 5.1% missing, and the missingness is *scattered*:
mostly 1995–2004, and only a handful of countries (IRL, AUT, BEL, LUX, DEU, GRC).
This is imputable without inventing data.

## 4. Loader verification

We verify that `src/preprocessing_extra.py` produces a clean country-year
panel for the current account balance.

**Note on shape.** The pivot in `load_extra_panel()` drops country-years
where current account is missing, producing 794 rows instead of 837.
These 43 rows are re-introduced as NaN by the merge in the next step
and imputed with country medians (fit on training rows only).

In [19]:
import sys
from pathlib import Path
current = Path.cwd()
project_root = current.parent if current.name == "notebooks" else current
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import preprocessing_extra as ppe

extra = ppe.load_extra_panel()
print("Shape:", extra.shape)
print("Columns:", extra.columns.tolist())
print("Year range:", extra["Year"].min(), "–", extra["Year"].max())
print("Countries:", extra["Country Code"].nunique())
print("Missing values:", extra["current_account_percent_gdp"].isna().sum())
print()
print(extra.head())

Shape: (794, 3)
Columns: ['Country Code', 'Year', 'current_account_percent_gdp']
Year range: 1995 – 2025
Countries: 27
Missing values: 0

  Country Code  Year  current_account_percent_gdp
0          AUT  2005                     1.985316
1          AUT  2006                     3.292509
2          AUT  2007                     3.813488
3          AUT  2008                     4.496157
4          AUT  2009                     2.580511


## 5. Merge diagnostic

We merge the extra panel onto the canonical model dataset (`model_data_v2.csv`)
to verify the join and see where the current account values are missing.

In [20]:
import pandas as pd
from src import config

# Load both panels
model_df = pd.read_csv(config.MODEL_DATA_PATH)
extra_df = ppe.load_extra_panel()

print("Model dataset:", model_df.shape)
print("Extra panel:", extra_df.shape)
print()

# Merge
merged = model_df.merge(
    extra_df,
    on=["Country Code", "Year"],
    how="left"
)
print("After merge:", merged.shape)
print()

# How many rows have NaN in current account?
print("Rows with missing current_account_percent_gdp:", 
      merged["current_account_percent_gdp"].isna().sum())
print()

# Which split are they in?
missing = merged[merged["current_account_percent_gdp"].isna()]
print("Missing by split:")
print(missing["dataset_split"].value_counts())
print()
print("Missing by country:")
print(missing["Country Code"].value_counts().to_string())

Model dataset: (810, 19)
Extra panel: (794, 3)

After merge: (810, 20)

Rows with missing current_account_percent_gdp: 36

Missing by split:
dataset_split
train    36
Name: count, dtype: int64

Missing by country:
Country Code
AUT    10
IRL    10
BEL     7
DEU     4
LUX     4
GRC     1


### Merge result — clean

- 810 rows in the merged dataset (same as the model dataset).
- 36 rows have missing current account — **all in the training split**.
- The test set is fully covered.
- Missing countries are mostly IRL, AUT, BEL, DEU, LUX — the same pattern we
  saw above.

**Implication:** the test set will never need imputation. Any imputation
we do happens only on training rows, using medians computed from training
rows. No leakage.

In [21]:
from src import preprocessing_extra as ppe

df_ext = ppe.merge_extra_features(pd.read_csv(config.MODEL_DATA_PATH))

print("Shape:", df_ext.shape)
print("New columns:", [c for c in df_ext.columns if c not in model_df.columns])
print()
print("Missing values:")
print(df_ext[[ppe.CURRENT_ACCOUNT_COL, ppe.CURRENT_ACCOUNT_LAG_COL]]
      .isna().sum())
print()
print("Split counts:")
print(df_ext["dataset_split"].value_counts())

Shape: (810, 21)
New columns: ['current_account_percent_gdp', 'current_account_lag_1']

Missing values:
current_account_percent_gdp    0
current_account_lag_1          0
dtype: int64

Split counts:
dataset_split
train    675
test     135
Name: count, dtype: int64


## Summary

- **5 candidate indicators tested.**
- **4 rejected** on coverage grounds (credit, broad money, terms of trade)
  or redundancy (GDP per capita growth).
- **1 viable** — current account balance — with 5.1% missing, all in
  the training period.
- Loader `src/preprocessing_extra.py` produces a clean merge:
  810 rows, 0 NaNs after imputation, test set untouched.

**Next step:** `08_feature_expansion_experiment.ipynb` — test whether
adding current account balance (and its lag) improves the tuned XGBoost
model on the held-out test set.